# Controllability analysis

This notebook is separate from the study-specific notebooks and focuses on controllability as a meta-metric built on top of the existing study metrics and invariance/control-variant caches.

Conceptual framing:

- `delta_C = B(ctrl_on) - B(ctrl_off)` per case
- aggregate with median (default) or mean
- bootstrap confidence intervals over paired evaluation units
- inspect robustness of controllability under paraphrase / ordering / pressure variants

See also:

- `docs/metrics/CONTROLLABILITY_META_METRIC.md`
- *Large-Language-Model Reasoning Failures* (`arXiv:2602.06176`)

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

from reliable_clinical_benchmark.invariance_analysis import VariantSpec, run_controllability_comparison

RUNTIME_ROOT = Path.cwd().resolve().parents[0]
DATA_ROOT = RUNTIME_ROOT / "data" / "frozen_splits" / "v5"
OUTPUT_PATH = RUNTIME_ROOT / "metric-results" / "controllability" / "example_controllability.json"

STUDY = "study_b"
BASE_CACHE = RUNTIME_ROOT / "results" / "qwen3-lmstudio" / "study_b_generations.jsonl"
VARIANTS = [
    VariantSpec(
        tag="mild",
        cache_path=RUNTIME_ROOT / "results" / "qwen3-lmstudio" / "study_b_invariance_mild.jsonl",
        variant_type="control",
        intensity=1.0,
    ),
    VariantSpec(
        tag="strong",
        cache_path=RUNTIME_ROOT / "results" / "qwen3-lmstudio" / "study_b_invariance_strong.jsonl",
        variant_type="control",
        intensity=3.0,
    ),
]

print("Update STUDY / BASE_CACHE / VARIANTS before executing the comparison cell.")

In [ ]:
payload = run_controllability_comparison(
    study=STUDY,
    base_cache=BASE_CACHE,
    variants=VARIANTS,
    data_root=DATA_ROOT,
    aggregation="median",
    n_resamples=1000,
    seed=42,
)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(OUTPUT_PATH)
payload

In [ ]:
records = []
for variant in payload["variants"]:
    for metric_name, metric in variant["metrics"].items():
        records.append(
            {
                "tag": variant["tag"],
                "variant_type": variant["variant_type"],
                "intensity": variant["intensity"],
                "metric": metric_name,
                "n_pairs": metric["n_pairs"],
                "base": metric["base"],
                "variant": metric["variant"],
                "delta_c": metric["delta_c"],
                "ci_low": metric["ci_low"],
                "ci_high": metric["ci_high"],
            }
        )

summary_df = pd.DataFrame(records)
display(summary_df.sort_values(["metric", "intensity", "tag"]))

In [ ]:
def plot_controllability(frame: pd.DataFrame, metric_name: str) -> None:
    subset = frame[frame["metric"] == metric_name].copy().sort_values(["intensity", "tag"])
    if subset.empty:
        print(f"No controllability rows for {metric_name}")
        return

    x = list(range(len(subset)))
    labels = [f"{tag}\n{intensity}" for tag, intensity in zip(subset["tag"], subset["intensity"])]
    lower = subset["delta_c"] - subset["ci_low"]
    upper = subset["ci_high"] - subset["delta_c"]

    plt.figure(figsize=(10, 4))
    plt.errorbar(x, subset["delta_c"], yerr=[lower, upper], fmt="o")
    plt.axhline(0.0, color="black", linestyle="--", linewidth=1)
    plt.xticks(x, labels)
    plt.ylabel("delta_C")
    plt.title(f"Controllability deltas: {metric_name}")
    plt.tight_layout()
    plt.show()


for metric_name in summary_df["metric"].unique():
    plot_controllability(summary_df, metric_name)

if payload.get("sensitivity_curves"):
    display(payload["sensitivity_curves"])